# 분주한 승강씨 — AI 다이얼로그 생성

**한 셀로 끝.** 한국어 LLM 로딩 → Gradio UI 띄움 → 장면 입력 → JSON 다이얼로그 출력.

**런타임 설정**: 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU**.

**모델**: Qwen2.5-7B-Instruct (한국어 우수, 7B 크기로 T4 에서 무리 없음).

**플로우**:
1. 아래 셀 실행 → 5~10분 모델 로딩 → Gradio 링크 (`https://xxx.gradio.live`) 클릭
2. 장면 입력 (예: "멘토가 신입에게 정원 풀 상황 가르치는 대화")
3. "생성" 클릭 → 다이얼로그 JSON 출력
4. 마음에 들면 ID 정하고 "dialog.json 형식으로 복사" 클릭 → 클립보드
5. CMS (`/cms.html` 다이얼로그 탭) 또는 `data/dialog.json` 에 붙여넣기

In [ ]:
# ============================================================
# 원셀: 의존성 → 모델 → Gradio UI
# ============================================================

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'bitsandbytes', 'gradio'], check=True)

import torch, json, re
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr

# ── 1) 캐릭터 / 게임 설정 (data/characters.json 과 일치) ──
CHARACTERS = {
    'mentor':   { 'displayName': '선임 (멘토)', 'persona': '베테랑 운영진. 비꼬는 말투, 시니컬. 가르치는 톤이지만 직설적. 시니어 매니저' },
    'owner':    { 'displayName': '사장님',       'persona': '50대 후반 빌딩 오너. 압박/평가 어조. 짧고 무겁게 말함. 결과만 보는 사람' },
    'player':   { 'displayName': '신입 매니저',  'persona': '플레이어 자신. 신입. 짧은 반응 위주 ("네...", "...열심히 하겠습니다"). 가끔 의문 던짐' },
    'narrator': { 'displayName': '내레이터',     'persona': '3인칭. 장면 묘사용. 캐릭터 대사 X' },
}

GAME_PREMISE = '''
게임 "분주한 승강씨": 빌딩 운영자(player)가 엘리베이터 운영 정책을 짜서 하루 트래픽을 관리하는 로그라이크.
- 엘베를 직접 조작하지 않고 "정책"만 설정
- 손님이 너무 오래 기다리면 "불만" → 5명 동시면 게임 오버
- 페이즈: 출근(morning) → 근무(work) → 점심(lunch) → 퇴근(evening) → 야간(night) → 1일 종료
- 골드는 하루 종료 시 층별 이용자 수 × 단가로 정산
- 멘토가 신입에게 운영 노하우 가르치는 톤. 일상 드라마 + 시니컬.
'''.strip()

OUTPUT_FORMAT_EXAMPLE = '''
출력은 반드시 이 JSON 배열 형식 (말풍선 하나 = 한 객체). 다른 텍스트는 출력 X:

[
  { "speaker": "mentor", "text": "손님 머리 위 게이지 보여? 그게 불만이야." },
  { "speaker": "player", "text": "빨개진 손님은 어떻게 되는 건데요?" },
  { "speaker": "mentor", "text": "불만 가득 차면 그날로 끝이야. 게임 오버.", "portrait": "smirk" }
]

규칙:
- speaker 는 narrator, mentor, owner, player 중 하나
- text 는 한국어. 말풍선당 ~40자 이내 권장
- portrait 는 옵션. "smirk"(시니컬), "worried"(걱정), "angry"(화남) 등 표정 키. 생략 가능
- 자연스럽게 mentor 와 player 가 번갈아. owner 는 가끔 등장. narrator 는 장면 시작/전환 시 1줄.
- 5~10줄이 적정
'''.strip()

# ── 2) 모델 로딩 (Qwen2.5-7B-Instruct, 4-bit quantize for T4) ─
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
print(f'{MODEL_ID} 로딩 중... (5~10분, 최초 한 번)')

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.eval()
print('모델 로딩 완료')

# ── 3) 생성 함수 ─────────────────────────────────
def build_system_prompt():
    char_block = '\n'.join([f"- {k} ({v['displayName']}): {v['persona']}" for k, v in CHARACTERS.items()])
    return f'''당신은 한국 인디 게임의 시나리오 작가입니다.

[게임 설정]
{GAME_PREMISE}

[등장인물]
{char_block}

[출력 형식]
{OUTPUT_FORMAT_EXAMPLE}

사용자가 장면을 요청하면 위 형식의 JSON 배열만 출력합니다. 설명 문구 X.
'''

def extract_json(text):
    '''LLM 출력에서 JSON 배열만 추출 (앞뒤 잡담 제거).'''
    # ``` 코드블럭 제거
    m = re.search(r'```(?:json)?\s*(\[.*?\])\s*```', text, re.DOTALL)
    if m: return m.group(1)
    # 첫 [ ... ] 만 추출
    start = text.find('[')
    if start == -1: return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '[': depth += 1
        elif text[i] == ']':
            depth -= 1
            if depth == 0: return text[start:i+1]
    return None

def generate_dialog(scene_prompt, num_lines=8, temperature=0.85):
    messages = [
        { 'role': 'system', 'content': build_system_prompt() },
        { 'role': 'user', 'content': f'다음 장면의 다이얼로그를 {num_lines}줄 내외로 작성해줘:\n\n{scene_prompt}' },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    json_str = extract_json(response)
    if not json_str:
        return None, response, f'❌ JSON 추출 실패. raw:\n{response[:500]}'
    try:
        parsed = json.loads(json_str)
        return parsed, response, f'✓ {len(parsed)} 줄 생성'
    except json.JSONDecodeError as e:
        return None, response, f'❌ JSON 파싱 실패: {e}\nraw:\n{json_str[:500]}'

# ── 4) Gradio UI ────────────────────────────────
def ui_generate(scene, num_lines, temp):
    parsed, raw, status = generate_dialog(scene, int(num_lines), float(temp))
    pretty = json.dumps(parsed, ensure_ascii=False, indent=2) if parsed else raw
    # 미리보기 (대화 형식)
    preview = ''
    if parsed:
        for line in parsed:
            sp = line.get('speaker', '?')
            name = CHARACTERS.get(sp, {}).get('displayName', sp)
            text = line.get('text', '')
            preview += f'**{name}**: {text}\n\n'
    return status, preview, pretty

def ui_format_for_dialog_json(script_id, raw_json):
    if not script_id or not raw_json.strip():
        return '❌ script ID + JSON 필요'
    try:
        parsed = json.loads(raw_json)
    except json.JSONDecodeError as e:
        return f'❌ JSON 오류: {e}'
    wrapped = { script_id: parsed }
    return json.dumps(wrapped, ensure_ascii=False, indent=2)

with gr.Blocks(title='Story Generator', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 분주한 승강씨 — 다이얼로그 생성기')
    gr.Markdown('장면을 한 줄로 설명하면 JSON 다이얼로그를 만듭니다. 마음에 들면 ID 정하고 dialog.json 에 붙여넣기.')

    with gr.Row():
        with gr.Column(scale=2):
            scene = gr.Textbox(label='장면 설명', lines=3, placeholder='예: 멘토가 신입에게 정원 가득 찬 엘베의 정책 우선순위를 가르치는 대화. 점심시간 직전, 살짝 압박감 있는 분위기.')
        with gr.Column(scale=1):
            num_lines = gr.Slider(3, 15, value=8, step=1, label='줄 수')
            temp = gr.Slider(0.3, 1.2, value=0.85, step=0.05, label='Temperature (높을수록 다양)')
    btn = gr.Button('🎬 다이얼로그 생성', variant='primary', size='lg')
    status = gr.Markdown()
    with gr.Row():
        preview = gr.Markdown(label='미리보기')
        raw = gr.Code(label='JSON 출력', language='json', lines=20)
    btn.click(ui_generate, [scene, num_lines, temp], [status, preview, raw])

    gr.Markdown('### dialog.json 형식으로 변환')
    with gr.Row():
        script_id = gr.Textbox(label='스크립트 ID', placeholder='예: scene-busy-lunch')
        format_btn = gr.Button('📋 dialog.json 형식으로 묶기')
    wrapped = gr.Code(label='dialog.json 에 붙여넣을 텍스트', language='json', lines=10)
    format_btn.click(ui_format_for_dialog_json, [script_id, raw], wrapped)

demo.launch(share=True)